## **Tareas a resolver:**

**Clasificación binaria: mentira o verdad**

**Predicción del hablante: de qué país es el mensaje**

Antes de empezar a resolver las tareas vamos a intentar arreglar el problema del desbalance de clases

**1. Carga y análisis inicial del dataset**
- Cargar datos
- Expandir mensajes
- Analizar distribución de clases
- Identificar desbalance

In [1]:
import pandas as pd
import json

data = pd.read_parquet('data/train_preprocessed.parquet')
df_expanded = data.explode(["messages", "sender_labels", "receiver_labels"])
df_expanded.head()

,messages,sender_labels,receiver_labels,speakers,receivers,absolute_message_index,relative_message_index,seasons,years,game_score,game_score_delta,players,game_id,text_clean,tokens,lemmas
0,Germany!\n\nJust the person I want to speak wi...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,germany just the person i want to speak with i...,"['germany', 'person', 'want', 'speak', 'somewh...","['germany', 'person', 'want', 'speak', 'somewh..."
1,"You've whet my appetite, Italy. What's the sug...",True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,youve whet my appetite italy whats the suggestion,"['ve', 'whet', 'appetite', 'italy', 's', 'sugg...","['ve', 'whet', 'appetite', 'italy', 's', 'sugg..."
2,It seems like there are a lot of ways that cou...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,it seems like there are a lot of ways that cou...,"['like', 'lot', 'ways', 'wrong', 'nt', 'france...","['like', 'lot', 'way', 'wrong', 'not', 'france..."
3,"Yeah, I can’t say I’ve tried it and it works, ...",True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,yeah i can t say i ve tried it and it works ca...,"['yeah', 't', 've', 'tried', 'works', 'cause',...","['yeah', 't', 've', 'try', 'work', 'cause', 'v..."
4,I am just sensing that you don’t like this ide...,True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,i am just sensing that you don t like this ide...,"['sensing', 'don', 't', 'like', 'idea', 'shall...","['sense', 'don', 't', 'like', 'idea', 'shall',..."


**2. Análisis del desbalance de clases**
- Distribución de `sender_labels`
- Distribución de `receiver_labels`

In [2]:
print("Distribución sender_labels:")
print(df_expanded["sender_labels"].value_counts())
print("\nDistribución receiver_labels:")
print(df_expanded["receiver_labels"].value_counts())

Distribución sender_labels:
sender_labels
True     11372
False      522
Name: count, dtype: int64

Distribución receiver_labels:
receiver_labels
True            10390
NOANNOTATION      989
False             515
Name: count, dtype: int64


**3. Corrección del desbalance de clases**
- Uso de *class weights*
- Preparación para entrenamiento

In [3]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# -------- sender_labels (binario) --------
y_sender = df_expanded["sender_labels"]

sender_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_sender),
    y=y_sender
)

print("\nClass weights para sender_labels:")
print(sender_class_weights)


# -------- receiver_labels (multiclase) --------
y_receiver = df_expanded["receiver_labels"]

receiver_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_receiver),
    y=y_receiver
)

print("\nClass weights para receiver_labels:")
print(receiver_class_weights)


Class weights para sender_labels:
[11.39272031  0.52295111]

Class weights para receiver_labels:
[7.69838188 4.00876306 0.38158486]


**Resultados**

Durante el entrenamiento de los modelos, utilizaremos estos pesos para que la función de pérdida multiplique el error de cada ejemplo por el peso correspondiente a su clase. De esta manera, el optimizador ajustará los parámetros del modelo para minimizar la pérdida ponderada, lo que mejora significativamente la capacidad del modelo para predecir correctamente las clases minoritarias.

In [15]:
# ======================================
# Preparación de vectores Word2Vec por mensaje
# ======================================

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from tqdm import tqdm
import ast

tqdm.pandas()

# ------------------------------
# Cargar DataFrame y Word2Vec
# ------------------------------
df = pd.read_parquet("data/train_preprocessed.parquet")
w2v = Word2Vec.load("diplomacy/models/embeddings/word2vec.model")

# ------------------------------
# Convertir tokens de string a lista (si es necesario)
# ------------------------------
if isinstance(df["tokens"].iloc[0], str):
    df["tokens"] = df["tokens"].apply(ast.literal_eval)

# ------------------------------
# Comprobación de tokens OOV
# ------------------------------
all_tokens = df["tokens"].explode()
in_vocab = all_tokens.apply(lambda w: w in w2v.wv)

print("Total de tokens:", len(all_tokens))
print("Tokens con embedding:", in_vocab.sum(), f"({in_vocab.mean()*100:.2f}%)")
print("Tokens fuera del vocabulario:", len(all_tokens) - in_vocab.sum())

# ------------------------------
# Función para generar vector promedio por mensaje
# ------------------------------
def message_to_vector(tokens, model):
    """
    Convierte una lista de tokens en un vector promedio usando Word2Vec.
    Tokens fuera del vocabulario se ignoran.
    """
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# ------------------------------
# Generar vectores para todos los mensajes
# ------------------------------
df["w2v_vector"] = df["tokens"].progress_apply(lambda tokens: message_to_vector(tokens, w2v))

# ------------------------------
# Verificar resultado
# ------------------------------
print("Primer vector (primeros 5 valores):", df["w2v_vector"].iloc[0][:5])
print("Shape de un vector:", df["w2v_vector"].iloc[0].shape)


Total de tokens: 98511
Tokens con embedding: 90643 (92.01%)
Tokens fuera del vocabulario: 7868


100%|██████████████████████████████████████████████████████| 11894/11894 [00:00<00:00, 36967.31it/s]

Primer vector (primeros 5 valores): [ 0.02355975 -0.10959967 -0.09957806  0.17633413  0.06859857]
Shape de un vector: (200,)


## **Shallow ML**
En esta sección vamos a intentar resolver las tareas con técnicas de shallow ML

### **1. Clasificación binaria**
   

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
import joblib

# --------------------------
# 1. Cargar datos
# --------------------------
X_tfidf = load_npz("diplomacy/models/representations/X_tfidf_train.npz")
X_bow   = load_npz("diplomacy/models/representations/X_bow_train.npz")

df = pd.read_parquet("data/train_preprocessed.parquet")

# Convertir etiquetas sender_labels a binario 0/1
y = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

print("Distribución de clases:")
print(y.value_counts())

# --------------------------
# 2. Pesos de clase
# --------------------------
class_weight_sender = {
    0: 11.39272031,   # False = minoritaria
    1: 0.52295111     # True = mayoritaria
}

# Para XGBoost usamos scale_pos_weight como recomendación oficial
pos_weight = class_weight_sender[0] / class_weight_sender[1]
print("\nscale_pos_weight =", pos_weight)

# --------------------------
# 3. División train/val
# --------------------------
X_train_tfidf, X_val_tfidf, y_train, y_val = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

X_train_bow, X_val_bow, _, _ = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)

# ==================================================
# 4. LOGISTIC REGRESSION (TF-IDF)
# ==================================================
lr = LogisticRegression(
    max_iter=3000,
    class_weight=class_weight_sender
)

lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_val_tfidf)

print("\n=== Logistic Regression (TF-IDF) ===")
print(classification_report(y_val, y_pred_lr))

joblib.dump(lr, "diplomacy/models/shallow/logreg_tfidf_weighted.joblib")

# ==================================================
# 5. LINEAR SVM (BoW)
# ==================================================
svm = LinearSVC(
    class_weight=class_weight_sender,
    max_iter=3000
)

svm.fit(X_train_bow, y_train)
y_pred_svm = svm.predict(X_val_bow)

print("\n=== Linear SVM (BoW) ===")
print(classification_report(y_val, y_pred_svm))

joblib.dump(svm, "diplomacy/models/shallow/svm_bow_weighted.joblib")

# ==================================================
# 6. XGBOOST (TF-IDF) – con oversampling para evitar collapse
# ==================================================
print("\nAplicando oversampling SOLO para XGBoost...")

ros = RandomOverSampler(random_state=42)
X_train_tfidf_bal, y_train_bal = ros.fit_resample(X_train_tfidf, y_train)

print("Distribución balanceada para XGBoost:")
print(pd.Series(y_train_bal).value_counts())

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=pos_weight,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train_tfidf_bal, y_train_bal)
y_pred_xgb = xgb.predict(X_val_tfidf)

print("\n=== XGBoost (TF-IDF + Oversampling + scale_pos_weight) ===")
print(classification_report(y_val, y_pred_xgb))

joblib.dump(xgb, "diplomacy/models/shallow/xgb_tfidf_weighted.joblib")

Distribución de clases:
sender_labels
1    11372
0      522
Name: count, dtype: int64

scale_pos_weight = 21.78544053573191

=== Logistic Regression (TF-IDF) ===
              precision    recall  f1-score   support

           0       0.11      0.22      0.15       104
           1       0.96      0.92      0.94      2275

    accuracy                           0.89      2379
   macro avg       0.54      0.57      0.55      2379
weighted avg       0.93      0.89      0.91      2379


=== Linear SVM (BoW) ===
              precision    recall  f1-score   support

           0       0.07      0.09      0.08       104
           1       0.96      0.95      0.95      2275

    accuracy                           0.91      2379
   macro avg       0.52      0.52      0.52      2379
weighted avg       0.92      0.91      0.92      2379


Aplicando oversampling SOLO para XGBoost...
Distribución balanceada para XGBoost:
sender_labels
1    9097
0    9097
Name: count, dtype: int64

=== XGBoost (T

['diplomacy/models/shallow/xgb_tfidf_weighted.joblib']

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from scipy.sparse import load_npz, hstack
from imblearn.over_sampling import SMOTE
from gensim.models import Word2Vec
from sklearn.preprocessing import StandardScaler
import joblib

# ======================================
# 1. CARGA DE DATOS Y ETIQUETAS
# ======================================
print("Cargando datos...")
X_tfidf = load_npz("diplomacy/models/representations/X_tfidf_train.npz")
df = pd.read_parquet("data/train_preprocessed.parquet")

# Etiquetas en formato binario
y = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

# Tus class weights calculados previamente
sender_class_weights = {
    0: 11.39272031,
    1: 0.52295111
}

# Cargar Word2Vec
w2v = Word2Vec.load("diplomacy/models/embeddings/word2vec.model")

# ======================================
# 2. GENERAR WORD2VEC PROMEDIADO POR TEXTO
# ======================================
def text_to_w2v(tokens):
    tokens = eval(tokens) if isinstance(tokens, str) else tokens
    vecs = [w2v.wv[word] for word in tokens if word in w2v.wv]
    if len(vecs) == 0:
        return np.zeros(w2v.vector_size)
    return np.mean(vecs, axis=0)

print("Generando embeddings Word2Vec...")
w2v_features = np.vstack(df["tokens"].apply(text_to_w2v).values)

# Escalado para combinar con TF-IDF
scaler = StandardScaler()
w2v_features_scaled = scaler.fit_transform(w2v_features)

# Convertir a sparse y concatenar con TF-IDF
w2v_sparse = np.nan_to_num(w2v_features_scaled)
X_combined = hstack([X_tfidf, w2v_sparse])

# ======================================
# 3. TRAIN/VAL SPLIT
# ======================================
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# ======================================
# 4. SMOTE (mejor que RandomOverSampler)
# ======================================
print("Aplicando SMOTE...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Distribución tras SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# ======================================
# 5. GRIDSEARCH PARA LOGISTIC REGRESSION
# ======================================
print("\nBuscando mejores hiperparámetros (LogReg)...")

params = {
    "C": [0.1, 1, 5],
    "penalty": ["l2"],
    "solver": ["liblinear", "lbfgs"],
    "class_weight": [
        None,
        "balanced",
        sender_class_weights
    ]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=3000),
    param_grid=params,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_smote, y_train_smote)

print("Mejores parámetros:", grid.best_params_)

best_lr = grid.best_estimator_
y_pred_lr = best_lr.predict(X_val)

print("\nResultados Logistic Regression + SMOTE + Word2Vec + GridSearch:")
print(classification_report(y_val, y_pred_lr))

joblib.dump(best_lr, "diplomacy/models/shallow/logreg_advanced.joblib")

# ======================================
# 6. BUSCAR UMBRAL ÓPTIMO PARA CLASE 0
# ======================================
print("\nBuscando umbral óptimo para clase minoritaria...")

y_probs = best_lr.predict_proba(X_val)[:, 1]

best_thr, best_f1 = 0, 0
for thr in np.arange(0.1, 0.9, 0.05):
    pred = (y_probs >= thr).astype(int)
    f1 = f1_score(y_val, pred, pos_label=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f"Mejor umbral = {best_thr:.2f} con F1(minoría) = {best_f1:.3f}")

final_pred = (y_probs >= best_thr).astype(int)

print("\nResultados Logistic Regression con umbral optimizado:")
print(classification_report(y_val, final_pred))

Cargando datos...
Generando embeddings Word2Vec...
Aplicando SMOTE...
Distribución tras SMOTE:
sender_labels
1    9097
0    9097
Name: count, dtype: int64

Buscando mejores hiperparámetros (LogReg)...
Mejores parámetros: {'C': 5, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}

Resultados Logistic Regression + SMOTE + Word2Vec + GridSearch:
              precision    recall  f1-score   support

           0       0.14      0.15      0.15       104
           1       0.96      0.96      0.96      2275

    accuracy                           0.92      2379
   macro avg       0.55      0.56      0.55      2379
weighted avg       0.93      0.92      0.92      2379


Buscando umbral óptimo para clase minoritaria...
Mejor umbral = 0.55 con F1(minoría) = 0.160

Resultados Logistic Regression con umbral optimizado:
              precision    recall  f1-score   support

           0       0.14      0.18      0.16       104
           1       0.96      0.95      0.96      2275

 

### **2. Predicción del hablante**


In [17]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# --------------------------
# 1. Cargar dataset preprocesado
# --------------------------
df = pd.read_parquet("data/train_preprocessed.parquet")

# --------------------------
# 2. Flatten speaker
# --------------------------
def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    elif isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        return x.strip('[]').replace("'", "").split(',')[0].strip()
    else:
        return str(x)

df['speakers'] = df['speakers'].apply(flatten_speaker)
y_speaker = df['speakers']
print("Clases finales:", y_speaker.nunique())

# --------------------------
# 3. Cargar representaciones dispersas
# --------------------------
X_bow = sp.load_npz("diplomacy/models/representations/X_bow_train.npz")[df.index]
X_tfidf = sp.load_npz("diplomacy/models/representations/X_tfidf_train.npz")[df.index]

representations = {
    "BOW": X_bow,
    "TF-IDF": X_tfidf
}

# --------------------------
# 4. Definir modelos a comparar
# --------------------------
models = {
    "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=1000, class_weight="balanced"),
    "Multinomial NB": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
}

# --------------------------
# 5. Entrenar y evaluar todos
# --------------------------
results = []

for rep_name, X in representations.items():
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_speaker, test_size=0.2, random_state=42, stratify=y_speaker
    )
    
    for model_name, clf in models.items():
        print(f"\nEntrenando {model_name} con {rep_name}...")
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Classification report
        report = classification_report(y_test, y_pred, output_dict=True, digits=4)
        accuracy = report["accuracy"]
        f1_macro = report["macro avg"]["f1-score"]
        precision_macro = report["macro avg"]["precision"]
        recall_macro = report["macro avg"]["recall"]
        
        # ROC AUC macro OvR
        try:
            y_test_bin = pd.get_dummies(y_test)
            y_pred_proba = clf.predict_proba(X_test)
            roc_auc = roc_auc_score(y_test_bin, y_pred_proba, average="macro", multi_class="ovr")
        except:
            roc_auc = np.nan  # algunos modelos no tienen predict_proba
        
        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "Accuracy": accuracy,
            "Precision": precision_macro,
            "Recall": recall_macro,
            "F1-Score": f1_macro,
            "ROC AUC (macro OvR)": roc_auc,
            "Observaciones": ""
        })

# --------------------------
# 6. Mostrar tabla comparativa
# --------------------------
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
df_results


Clases finales: 7

Entrenando Logistic Regression con BOW...

Entrenando Multinomial NB con BOW...

Entrenando Random Forest con BOW...

Entrenando Logistic Regression con TF-IDF...

Entrenando Multinomial NB con TF-IDF...

Entrenando Random Forest con TF-IDF...


C:\Users\Alba\Documents\NLP\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Alba\Documents\NLP\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Alba\Documents\NLP\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Representación,Modelo,Accuracy,Precision,Recall,F1-Score,ROC AUC (macro OvR),Observaciones
0,TF-IDF,Logistic Regression,0.336696,0.306508,0.312522,0.304094,0.700371,
1,BOW,Logistic Regression,0.326608,0.285968,0.288867,0.286346,0.675402,
2,BOW,Multinomial NB,0.377890,0.316654,0.284893,0.286268,0.687921,
3,BOW,Random Forest,0.356452,0.283536,0.241423,0.228996,0.667852,
4,TF-IDF,Random Forest,0.355191,0.275144,0.228449,0.206393,0.671700,
5,TF-IDF,Multinomial NB,0.363598,0.288579,0.208407,0.146863,0.682608,


## **CNNs o Redes Recurrentes**

En esta sección vamos a entrenar modelos CNN para resolver las tareas

### **1. Clasificación binaria**
   

In [18]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from gensim.models import Word2Vec, FastText
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import BertTokenizer, BertModel
import joblib

# --------------------------
# Paths y Config
# --------------------------
DATA_PATH = "data/train_preprocessed.parquet"
EMB_DIR = "diplomacy/models/embeddings"
OUT_DIR = "diplomacy/models/deep"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
EPOCHS = 5  # puedes aumentar si quieres
MAX_LEN = 120
EMB_DIM = 200  # Debe coincidir con Word2Vec/FastText

# --------------------------
# Cargar dataframe y etiquetas
# --------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].isin(["true","false","True","False"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true":1,"false":0})
le = LabelEncoder()
y = le.fit_transform(y_raw)

class_weights = compute_class_weight("balanced", classes=np.unique(y), y=y)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# --------------------------
# Funciones utilitarias
# --------------------------
def to_indices(tokens, token2idx):
    if isinstance(tokens, str):
        tokens = eval(tokens) if tokens.startswith("[") else tokens.split()
    idxs = [token2idx.get(tok, token2idx["<oov>"]) for tok in tokens]
    if len(idxs) >= MAX_LEN:
        return idxs[:MAX_LEN]
    else:
        return idxs + [token2idx["<pad>"]] * (MAX_LEN - len(idxs))

class DiplomacyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long) if isinstance(X, np.ndarray) else X
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, emb_matrix=None, freeze_emb=True, n_filters=100, kernel_sizes=[3,4,5], num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if emb_matrix is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.embedding.weight.requires_grad = not freeze_emb
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.fc = nn.Linear(n_filters*len(kernel_sizes), num_classes)
    def forward(self, x):
        e = self.embedding(x).permute(0,2,1)
        convs = [torch.relu(conv(e)) for conv in self.convs]
        pools = [torch.max(c, dim=2)[0] for c in convs]
        cat = torch.cat(pools, dim=1)
        return self.fc(cat)

def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val_f1 = 0
    best_state = None
    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
        # evaluación
        model.eval()
        y_trues, y_preds = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                logits = model(Xb)
                preds = logits.argmax(dim=1).cpu().numpy()
                y_preds.extend(preds)
                y_trues.extend(yb.cpu().numpy())
        val_f1 = f1_score(y_trues, y_preds, average="macro")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()
    if best_state:
        model.load_state_dict(best_state)
    return model

def evaluate_model(model, loader):
    model.eval()
    y_trues, y_preds = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            y_preds.extend(preds)
            y_trues.extend(yb.cpu().numpy())
    acc = accuracy_score(y_trues, y_preds)
    f1m = f1_score(y_trues, y_preds, average="macro")
    prec = f1_score(y_trues, y_preds, average="macro", labels=[0,1])
    recall = f1_score(y_trues, y_preds, average="macro", labels=[0,1])
    return acc, prec, recall, f1m

# --------------------------
# Embeddings: Word2Vec y FastText
# --------------------------
embedding_models = {
    "Word2Vec": Word2Vec.load(os.path.join(EMB_DIR,"word2vec.model")),
    "FastText": FastText.load(os.path.join(EMB_DIR,"fasttext.model"))
}

results = []

for emb_name, model in embedding_models.items():
    print(f"\nProcesando {emb_name} embeddings...")
    token2idx = {"<pad>":0,"<oov>":1}
    for tok in model.wv.key_to_index.keys():
        token2idx[tok] = len(token2idx)
    vocab_size = len(token2idx)
    
    X_seq = np.vstack(df["tokens"].apply(lambda t: np.array(to_indices(t, token2idx))).values)
    X_train, X_val, y_train_split, y_val_split = train_test_split(X_seq, y, test_size=0.2, random_state=42, stratify=y)
    
    train_loader = DataLoader(DiplomacyDataset(X_train, y_train_split), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(DiplomacyDataset(X_val, y_val_split), batch_size=BATCH_SIZE)
    
    # embedding matrix
    emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMB_DIM)).astype(np.float32)
    for tok, idx in token2idx.items():
        if tok in model.wv:
            emb_matrix[idx] = model.wv[tok]
    
    for freeze in [True, False]:
        cnn = CNNClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, emb_matrix=emb_matrix, freeze_emb=freeze)
        cnn = train_model(cnn, train_loader, val_loader)
        acc, prec, recall, f1m = evaluate_model(cnn, val_loader)
        results.append({
            "Representación": emb_name,
            "Modelo": f"CNN {'freeze' if freeze else 'fine-tune'}",
            "Accuracy": acc,
            "Precision": prec,
            "Recall": recall,
            "F1-Score": f1m,
            "Observaciones": ""
        })

# --------------------------
# Embeddings: BERT
# --------------------------
print("\nProcesando BERT embeddings...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(DEVICE)

class BertDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {k:v[idx] for k,v in self.encodings.items()}, self.labels[idx]

class BertClassifier(nn.Module):
    def __init__(self, bert_model, num_classes=2, freeze=True):
        super().__init__()
        self.bert = bert_model
        self.bert.requires_grad_(not freeze)
        self.fc = nn.Linear(bert_model.config.hidden_size, num_classes)
    def forward(self, batch):
        outputs = self.bert(**batch)
        pooled = outputs.pooler_output
        return self.fc(pooled)

X_train_text, X_val_text, y_train_split, y_val_split = train_test_split(df["messages"], y, test_size=0.2, stratify=y)
train_loader = DataLoader(BertDataset(X_train_text, y_train_split), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(BertDataset(X_val_text, y_val_split), batch_size=BATCH_SIZE)

for freeze in [True, False]:
    model = BertClassifier(bert_model, freeze=freeze).to(DEVICE)
    # Entrenamiento sencillo
    opt = torch.optim.Adam(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_f1 = 0
    best_state = None
    for epoch in range(EPOCHS):
        model.train()
        for batch, labels in train_loader:
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            labels = labels.to(DEVICE)
            opt.zero_grad()
            logits = model(batch)
            loss = criterion(logits, labels)
            loss.backward()
            opt.step()
        # eval
        model.eval()
        y_preds, y_trues = [], []
        with torch.no_grad():
            for batch, labels in val_loader:
                batch = {k:v.to(DEVICE) for k,v in batch.items()}
                labels = labels.to(DEVICE)
                logits = model(batch)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                y_preds.extend(preds)
                y_trues.extend(labels.cpu().numpy())
        val_f1 = f1_score(y_trues, y_preds, average="macro")
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict()
    if best_state:
        model.load_state_dict(best_state)
    acc = accuracy_score(y_trues, y_preds)
    f1m = f1_score(y_trues, y_preds, average="macro")
    prec = f1_score(y_trues, y_preds, average="macro", labels=[0,1])
    recall = f1_score(y_trues, y_preds, average="macro", labels=[0,1])
    results.append({
        "Representación": "BERT",
        "Modelo": f"CNN {'freeze' if freeze else 'fine-tune'}",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": recall,
        "F1-Score": f1m,
        "Observaciones": ""
    })

# --------------------------
# Mostrar tabla final
# --------------------------
df_results = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
df_results



Procesando Word2Vec embeddings...

Procesando FastText embeddings...

Procesando BERT embeddings...


,Representación,Modelo,Accuracy,Precision,Recall,F1-Score,Observaciones
0,BERT,CNN fine-tune,0.830181,0.515750,0.515750,0.515750,
1,BERT,CNN freeze,0.819672,0.502064,0.502064,0.502064,
2,Word2Vec,CNN fine-tune,0.845734,0.498003,0.498003,0.498003,
3,FastText,CNN fine-tune,0.833544,0.495897,0.495897,0.495897,
4,FastText,CNN freeze,0.780160,0.491820,0.491820,0.491820,
5,Word2Vec,CNN freeze,0.741488,0.484328,0.484328,0.484328,


### **2. Predicción del hablante**


In [2]:
"""
Predicción del hablante con Deep Learning (versión optimizada para CPU):
- LSTM
- CNN
- Embedding Word2Vec (si existe) o aleatorio
- Class Weights para el desequilibrio
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from gensim.models import Word2Vec
from tqdm import tqdm
import joblib

# ------------------------------------
# 1. Configuración
# ------------------------------------
DEVICE = "cpu"   # FORZAR CPU
BATCH_SIZE = 32  # reducir para acelerar CPU
EPOCHS = 8       # razonable para CPU
EMB_DIM = 200   
MAX_LEN = 120

EMB_DIR = "diplomacy/models/embeddings"
OUT_DIR = "diplomacy/models/deep_speaker_cpu"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------
# 2. Cargar dataset
# ------------------------------------
df = pd.read_parquet("data/train_preprocessed.parquet")

# ------------------------------------
# 2.1 Flatten igual que tu pipeline shallow
# ------------------------------------
def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    elif isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        return x.strip('[]').replace("'", "").split(',')[0].strip()
    else:
        return str(x)

df["speakers"] = df["speakers"].apply(flatten_speaker)
y = df["speakers"]

# ------------------------------------
# 3. Codificar labels
# ------------------------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)

print("Número de clases:", num_classes)

# ------------------------------------
# 4. Intentar cargar Word2Vec
# ------------------------------------
USE_W2V = True
w2v_path = os.path.join(EMB_DIR, "word2vec.model")

if USE_W2V and os.path.exists(w2v_path):
    w2v = Word2Vec.load(w2v_path)
    print("Word2Vec cargado:", len(w2v.wv))
else:
    USE_W2V = False
    print("No se encontró Word2Vec → Usando embedding aleatorio")

# ------------------------------------
# 5. Vocabulario
# ------------------------------------
token2idx = {"<pad>": 0, "<oov>": 1}

if USE_W2V:
    for word in w2v.wv.key_to_index.keys():
        token2idx[word] = len(token2idx)

vocab_size = len(token2idx)
print("Vocab size:", vocab_size)

# ------------------------------------
# 6. Convertir tokens → índices
# ------------------------------------
def to_indices(tokens):
    if isinstance(tokens, str):
        tokens = eval(tokens)
    idxs = [token2idx.get(tok, 1) for tok in tokens]
    if len(idxs) >= MAX_LEN:
        return idxs[:MAX_LEN]
    else:
        return idxs + [0] * (MAX_LEN - len(idxs))

X_seq = np.vstack(df["tokens"].apply(lambda t: np.array(to_indices(t))).values)

# ------------------------------------
# 7. train/test split
# ------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# ------------------------------------
# 8. Class weights
# ------------------------------------
cls_weights = compute_class_weight(
    "balanced", classes=np.unique(y_enc), y=y_enc
)
cls_weights = torch.tensor(cls_weights, dtype=torch.float32)
print("Class weights:", cls_weights)

# ------------------------------------
# 9. Dataset
# ------------------------------------
class SpeakerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = SpeakerDataset(X_train, y_train)
val_ds = SpeakerDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ------------------------------------
# 10. Embedding Matrix (CPU friendly)
# ------------------------------------
emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMB_DIM)).astype(np.float32)

if USE_W2V:
    for tok, idx in token2idx.items():
        if tok in w2v.wv:
            emb_matrix[idx] = w2v.wv[tok]

# ------------------------------------
# 11. Modelos optimizados para CPU
# ------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMB_DIM, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))

        self.lstm = nn.LSTM(
            EMB_DIM, 64, batch_first=True, bidirectional=True, dropout=0.1
        )
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        e = self.embedding(x)
        out, _ = self.lstm(e)
        out = out.mean(dim=1)
        return self.fc(out)

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMB_DIM, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))

        # Reducido para CPU
        self.convs = nn.ModuleList([
            nn.Conv1d(EMB_DIM, 64, k) for k in [3,4,5]
        ])

        self.fc = nn.Linear(64*3, num_classes)

    def forward(self, x):
        e = self.embedding(x).permute(0,2,1)
        convs = [torch.relu(conv(e)) for conv in self.convs]
        pools = [torch.max(c, dim=2)[0] for c in convs]
        cat = torch.cat(pools, dim=1)
        return self.fc(cat)

# ------------------------------------
# 12. Entrenador optimizado para CPU
# ------------------------------------
def train_model(model, name):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=cls_weights)

    best_f1 = 0
    best_state = None

    for epoch in range(EPOCHS):
        model.train()
        for Xb, yb in train_loader:
            opt.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()

        # eval
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                logits = model(Xb)
                preds = torch.argmax(logits, 1).numpy()
                y_pred.extend(preds)
                y_true.extend(yb.numpy())

        f1 = f1_score(y_true, y_pred, average="macro")
        print(f"[{name}] Epoch {epoch+1}/{EPOCHS} - F1_macro={f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(OUT_DIR, f"{name}.pt"))
    return model

# ------------------------------------
# 13. Entrenar ambos modelos
# ------------------------------------
print("\nEntrenando CNN...")
cnn = train_model(CNNClassifier(), "cnn_cpu")

print("\nEntrenando LSTM...")
lstm = train_model(LSTMClassifier(), "lstm_cpu")

# ------------------------------------
# 14. Evaluación final
# ------------------------------------
def evaluate(model, name):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            logits = model(Xb)
            preds = torch.argmax(logits, 1).numpy()
            y_pred.extend(preds)
            y_true.extend(yb.numpy())

    print(f"\n=== RESULTADOS {name} ===")
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1 macro:", f1_score(y_true, y_pred, average="macro"))

evaluate(cnn, "CNN")
evaluate(lstm, "LSTM")

joblib.dump(le, os.path.join(OUT_DIR, "label_encoder.joblib"))


Número de clases: 7
Word2Vec cargado: 2025
Vocab size: 2027
Class weights: tensor([1.0476, 0.5752, 1.6626, 1.2814, 0.5584, 1.5308, 2.0797])

Entrenando CNN...
[cnn_cpu] Epoch 1/8 - F1_macro=0.2196
[cnn_cpu] Epoch 2/8 - F1_macro=0.2505
[cnn_cpu] Epoch 3/8 - F1_macro=0.2759
[cnn_cpu] Epoch 4/8 - F1_macro=0.2935
[cnn_cpu] Epoch 5/8 - F1_macro=0.2938
[cnn_cpu] Epoch 6/8 - F1_macro=0.2851
[cnn_cpu] Epoch 7/8 - F1_macro=0.2840
[cnn_cpu] Epoch 8/8 - F1_macro=0.2884

Entrenando LSTM...


C:\Users\Oihane\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn(


[lstm_cpu] Epoch 1/8 - F1_macro=0.1342
[lstm_cpu] Epoch 2/8 - F1_macro=0.2035
[lstm_cpu] Epoch 3/8 - F1_macro=0.2220
[lstm_cpu] Epoch 4/8 - F1_macro=0.2522
[lstm_cpu] Epoch 5/8 - F1_macro=0.2534
[lstm_cpu] Epoch 6/8 - F1_macro=0.2521
[lstm_cpu] Epoch 7/8 - F1_macro=0.2551
[lstm_cpu] Epoch 8/8 - F1_macro=0.2616

=== RESULTADOS CNN ===
              precision    recall  f1-score   support

     austria       0.23      0.29      0.26       325
     england       0.48      0.45      0.46       591
      france       0.22      0.29      0.25       204
     germany       0.26      0.25      0.26       265
       italy       0.44      0.34      0.39       609
      russia       0.21      0.25      0.23       222
      turkey       0.18      0.18      0.18       163

    accuracy                           0.33      2379
   macro avg       0.29      0.29      0.29      2379
weighted avg       0.34      0.33      0.33      2379

Accuracy: 0.32618747372845736
F1 macro: 0.2883841061187325

=== RES

['diplomacy/models/deep_speaker_cpu\\label_encoder.joblib']

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader

# ======================================================
# 1. Cargar dataset y labels
# ======================================================

df = pd.read_parquet("data/train_preprocessed.parquet")

def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    elif isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        return x.strip('[]').replace("'", "").split(',')[0].strip()
    else:
        return str(x)

df["speakers"] = df["speakers"].apply(flatten_speaker)
y = df["speakers"]

le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)

# ======================================================
# 2. Cargar embeddings BERT ya generados
# ======================================================

X_bert = np.load("diplomacy/models/embeddings/bert_train.npy")
print("Shape BERT:", X_bert.shape)   # (N, 768)

# ======================================================
# 3. División train/val
# ======================================================
X_train, X_val, y_train, y_val = train_test_split(
    X_bert, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# ======================================================
# 4. Class weights
# ======================================================
cls_weights = compute_class_weight("balanced", classes=np.unique(y_enc), y=y_enc)
cls_weights = torch.tensor(cls_weights, dtype=torch.float32)

# ======================================================
# 5. Dataset
# ======================================================
class BertDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(BertDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(BertDataset(X_val, y_val), batch_size=32)

# ======================================================
# 6. Modelo MLP
# ======================================================
class BERT_MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = BERT_MLP()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(weight=cls_weights)

# ======================================================
# 7. Entrenamiento
# ======================================================
EPOCHS = 15
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    # Validación
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            y_pred.extend(torch.argmax(logits, dim=1).numpy())
            y_true.extend(yb.numpy())

    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"Epoch {epoch+1} - F1_macro={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_state = model.state_dict()

# Cargar mejor modelo
model.load_state_dict(best_state)

# ======================================================
# 8. Evaluación final
# ======================================================
print("\n=== RESULTADOS BERT_MLP ===")
print(classification_report(y_true, y_pred, target_names=le.classes_))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 macro:", f1)

Shape BERT: (11894, 768)
Epoch 1 - F1_macro=0.0980
Epoch 2 - F1_macro=0.0888
Epoch 3 - F1_macro=0.0958
Epoch 4 - F1_macro=0.0823
Epoch 5 - F1_macro=0.0971
Epoch 6 - F1_macro=0.0947
Epoch 7 - F1_macro=0.1179
Epoch 8 - F1_macro=0.1029
Epoch 9 - F1_macro=0.1252
Epoch 10 - F1_macro=0.1067
Epoch 11 - F1_macro=0.1441
Epoch 12 - F1_macro=0.0956
Epoch 13 - F1_macro=0.1084
Epoch 14 - F1_macro=0.1275
Epoch 15 - F1_macro=0.1289

=== RESULTADOS BERT_MLP ===
              precision    recall  f1-score   support

     austria       0.21      0.02      0.04       325
     england       0.25      0.21      0.23       591
      france       0.13      0.42      0.20       204
     germany       0.15      0.51      0.23       265
       italy       0.25      0.06      0.10       609
      russia       0.12      0.08      0.09       222
      turkey       0.00      0.00      0.00       163

    accuracy                           0.17      2379
   macro avg       0.16      0.19      0.13      2379
weighted